In [ ]:
# @title 1. Install Dependencies (Run Once)
# We use 'uv' because it is faster and resolves dependency conflicts better than pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install --system vllm lmcache pandas matplotlib numpy
!nvidia-smi # Verify you have a T4 GPU

downloading uv 0.10.6 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.12.12 environment at: /usr
Resolved 186 packages in 1.30s
Prepared 64 packages in 34.29s
Uninstalled 12 packages in 1.07s
Installed 64 packages in 972ms
 + aiofile==3.9.0
 + anthropic==0.84.0
 + apache-tvm-ffi==0.1.8.post2
 + astor==0.8.1
 + awscrt==0.31.2
 + blake3==1.0.8
 + caio==0.9.25
 + cbor2==5.8.0
 + compressed-tensors==0.13.0
 + cufile-python==0.2.0
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.14.0
 + fastar==0.8.0
 + flashinfer-python==0.6.3
 + gguf==0.17.1
 + grpcio-reflection==1.78.0
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + lmcache==0.3.14
 + loguru==0.7.3

In [ ]:
# @title 2. Create LMCache Config (Run Before Simulation)
# Create a dedicated directory for our L3 Cache (Disk Tier)
import os
os.makedirs("/content/lmcache_store", exist_ok=True)

# Create the LMCache Config that forces Disk Storage
import yaml

config = {
    "chunk_size": 256,
    "local_cpu": True,
    "max_local_cpu_size": 5.0, # 5GB RAM Cache
    "local_disk": True,
    "remote_url": "file:///content/lmcache_store", # The Persistent L3 Tier
    "remote_serde": "naive",
    "save_decode_cache": True # Cache the generated output too
}

with open("lmcache_config.yaml", "w") as f:
    yaml.dump(config, f)

print("✅ LMCache configured. Storage Tier: /content/lmcache_store")

✅ LMCache configured. Storage Tier: /content/lmcache_store


In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

os.environ["LMCACHE_CONFIG_FILE"] = "lmcache_config.yaml"

# 3. Initialize Engine
from vllm import LLM, SamplingParams
print("🚀 Initializing vLLM Engine...")
llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    kv_transfer_config={"kv_connector": "LMCacheConnectorV1", "kv_role": "kv_both"},
    enforce_eager=True,
    gpu_memory_utilization=0.6,
    max_model_len=4096, # Support long docs
    disable_log_stats=True
)
sampling_params = SamplingParams(temperature=0, max_tokens=20)

# --- THE SIMULATION LOGIC ---
results = []

# --- 🛠️ UPDATED SIMULATION LOGIC WITH ADVANCED METRICS ---
import psutil
import shutil

# Helper 1: Get Disk Cache Size (L3 Tier)
def get_cache_size_mb(path):
    if not os.path.exists(path): return 0
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return round(total_size / (1024 * 1024), 2) # Convert bytes to MB

# Helper 2: Get RAM Usage (L2 Tier)
def get_ram_usage_gb():
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024 * 1024), 2) # GB

def run_test_advanced(scenario_name, prompt_list, clear_cache=False):
    print(f"\n--- 🧪 Running Advanced Test: {scenario_name} ---")

    # 1. Clear Cache if requested (Simulating Cold Start)
    if clear_cache:
        # Note: In a real script, we'd restart the engine.
        # Here we simulate 'Cold' by just logging it as the first run.
        pass

    for i, prompt in enumerate(prompt_list):
        # --- PRE-RUN METRICS ---
        start_ram = get_ram_usage_gb()
        start_disk = get_cache_size_mb("/content/lmcache_store")

        # --- INFERENCE ---
        t0 = time.perf_counter()
        outputs = llm.generate(prompt, sampling_params)
        t1 = time.perf_counter()

        # --- POST-RUN METRICS ---
        latency = t1 - t0
        prompt_tokens = len(outputs[0].prompt_token_ids)
        gen_tokens = len(outputs[0].outputs[0].token_ids)
        total_tokens = prompt_tokens + gen_tokens

        # 1. Throughput (How much work per second?)
        throughput = total_tokens / latency if latency > 0 else 0

        # 2. Normalized Latency (Cost per input token)
        # This allows comparing a 50-token query vs a 2000-token query fairly
        ms_per_token = (latency * 1000) / prompt_tokens if prompt_tokens > 0 else 0

        # 3. Determine Hit/Miss State
        # We assume first query is always Miss (Cold), subsequent are Hits (Warm)
        # unless it's the "No Context" scenario
        state = "Cold (Miss)" if i == 0 else "Warm (Hit)"
        if scenario_name == "3. No Context": state = "Cold (Miss)" # Random queries never hit

        print(f"   Query {i+1} ({state}): {latency:.2f}s | {throughput:.1f} tok/s | Disk: {start_disk}MB")

        results.append({
            "Scenario": scenario_name,
            "Query_ID": i + 1,
            "State": state,
            "Total Latency (s)": round(latency, 4),
            "Throughput (tok/s)": round(throughput, 2),
            "Cost (ms/token)": round(ms_per_token, 2),
            "L3 Disk Usage (MB)": start_disk,
            "L2 RAM Usage (GB)": start_ram,
            "Input Size (Tokens)": prompt_tokens
        })

# ==========================================
# 🧪 EXPERIMENT 1: SHARED PREFIX (Persona)
# ==========================================
# Short system prompt (~50 tokens)
prefix = "You are an expert Physiotherapist AI for RehabQuest. Answer briefly. "
prompts_1 = [prefix + "Exercise for back pain?", prefix + "Is ice good for swelling?", prefix + "Define posture."]
run_test_advanced("1. Shared Prefix", prompts_1, clear_cache=True)

# ==========================================
# 🧪 EXPERIMENT 2: SHARED DOCS (RAG)
# ==========================================
# Massive Context (~2000 tokens) - Simulating a Manual
doc = "RehabQuest Technical Manual: " + ("Geometric calculation of knee angles requires... " * 300)
prompts_2 = [doc + "How to calculate knee angle?", doc + "What is the formula?", doc + "Explain errors."]
run_test_advanced("2. Shared Docs (RAG)", prompts_2, clear_cache=True)

# ==========================================
# 🧪 EXPERIMENT 3: NO CONTEXT (Random)
# ==========================================
# Totally different prompts
prompts_3 = ["Who is Batman?", "Recipe for cake?", "Distance to Moon?"]
run_test_advanced("3. No Context", prompts_3, clear_cache=True)

# ==========================================
# 🧪 EXPERIMENT 4: MULTI-TURN CHAT
# ==========================================
# History grows: Q1 -> Q1+A1+Q2 -> Q1+A1+Q2+A2+Q3...
history = "User: Hello AI.\nAI: Hi there.\n"
prompts_4 = []
# Simulate 3 turns where context grows by 500 tokens each time
for i in range(3):
    history += f"User: Question {i}. " + ("context " * 50) + "\n"
    prompts_4.append(history)

run_test_advanced("4. Multi-Turn Chat", prompts_4, clear_cache=True)

# --- 📊 ANALYSIS & PLOTTING ---
df = pd.DataFrame(results)

# Calculate Speedup (Cold / Warm)
# We compare Query 1 (Cold) vs Query 2 (Warm) for shared scenarios
print("\n--- 🏆 FINAL RESULTS ---")
pivot = df.pivot_table(index="Scenario", columns="Query_ID", values="TTFT (s)")
pivot["Speedup (x)"] = pivot[1] / pivot[2]
print(pivot[[1, 2, "Speedup (x)"]])

# Plot
plt.figure(figsize=(12, 6))
scenarios = df["Scenario"].unique()
x = np.arange(len(scenarios))
width = 0.35

# Filter Query 1 (Cold) and Query 2 (Warm)
q1_times = df[df["Query_ID"] == 1].set_index("Scenario").reindex(scenarios)["TTFT (s)"]
q2_times = df[df["Query_ID"] == 2].set_index("Scenario").reindex(scenarios)["TTFT (s)"]

plt.bar(x - width/2, q1_times, width, label='Query 1 (Cold/Compute)', color='#FF6B6B')
plt.bar(x + width/2, q2_times, width, label='Query 2 (Warm/Cache)', color='#4ECDC4')

plt.ylabel('Time To First Token (s)')
plt.title('RehabQuest Inference: LMCache Impact by Query Type')
plt.xticks(x, scenarios)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

WARNING 02-26 17:37:15 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
["Who is the founder of Microsoft?",
"What is the chemical symbol for sodium?",
"How many players are there in a baseball team?",
"What year did World War II end?",
"What is the tallest species of tree?",
"Who wrote The Divine Comedy?",
"What is the hardest rock type?",
"How many hearts does a squid have?",
"What is the main ingredient in guacamole?",
"Who developed the polio vaccine?",
"What is the longest-running Broadway show?",
"What is the square of 25?",
"Who was the first woman to win a Nobel Prize?",
"What is the currency of Brazil?",
"What gas do plants absorb during photosynthesis?",
"Who directed the movie Jaws?",
"What is the largest internal organ in the human body?",
"How many elements are in the periodic table?",
"What is the freezing point of mercury in Celsius?",
"Who painted The School of Athens?",
"What is the smallest unit of life?",
"How many time zones are there in Russia?",
"What is the main language spoken in Argentina?",
"Who invented the diesel engine?",
"What is the diameter of Earth in kilometers?",
"What is the rarest blood type?",
"Who composed The Magic Flute?",
"What is the powerhouse of a computer?",
"How many sides does a dodecagon have?",
"What is the largest species of penguin?",
"Who discovered the planet Neptune?",
"What is the boiling point of nitrogen in Celsius?",
"What is the fastest bird in a dive?",
"Who wrote The Brothers Karamazov?",
"What is the largest artery in the human body?",
"How many keys are on a standard computer keyboard?",
"What is the smallest country by population?",
"Who invented the telescope?",
"What is the main component of natural gas?",
"How many amendments are in the U.S. Constitution?",
"What is the deepest lake in the world?",
"Who was the first emperor of China?",
"What is the currency of South Africa?",
"What is the longest bone in the arm?",
"Who discovered the circulation of blood?",
"What is the primary ingredient in hummus?",
"How many moons does Mars have?",
"What is the largest species of cat?",
"Who wrote The Picture of Dorian Gray?",
"What is the chemical formula for methane?",
"What is the highest-grossing film of all time?",
"Who invented the cotton gin?",
"What is the main source of vitamin D?",
"How many strings does a harp typically have?",
"What is the smallest unit of matter?",
"Who painted The Creation of Adam?",
"What is the largest airport in the world by area?",
"How many bones are in a newborn baby?",
"What is the longest reigning British monarch?",
"Who discovered insulin?",
"What is the main ingredient in miso soup?",
"How many planets are gas giants in our solar system?",
"What is the largest freshwater lake in Africa?",
"Who wrote The Alchemist?",
"What is the chemical symbol for potassium?",
"How many players are on a rugby union team?",
"What is the tallest waterfall in Africa?",
"Who invented the mechanical clock?",
"What is the main export of Saudi Arabia?",
"How many chromosomes does a human gamete contain?",
"What is the largest desert in Asia?",
"Who composed Swan Lake?",
"What is the smallest muscle in the human body?",
"How many colors are there in the visible spectrum?",
"What is the largest species of eagle?",
"Who discovered X-rays?",
"What is the boiling point of ethanol in Celsius?",
"How many rings are on the Olympic flag?",
"What is the longest river in South America?",
"Who invented the sewing machine?",
"What is the primary ingredient in pesto?",
"How many players are on a water polo team?",
"What is the largest organ in the digestive system?",
"Who wrote The Old Man and the Sea?",
"What is the chemical symbol for zinc?",
"How many bones are in the adult human skull?",
"What is the highest active volcano in the world?",
"Who developed the theory of evolution by natural selection?",
"What is the main ingredient in falafel?",
"How many faces does an icosahedron have?",
"What is the largest species of whale?",
"Who invented the barcode?",
"What is the smallest planet in our solar system?",
"How many players are on a cricket team?",
"What is the largest glacier in the world?",
"Who wrote The Count of Monte Cristo?",
"What is the chemical symbol for silver?",
"How many teeth does a shark typically have in its lifetime?",
"What is the highest mountain in South America?",
"Who discovered the law of planetary motion?",
"What is the primary ingredient in kimchi?",
"How many symphonies did Beethoven compose?",
"What is the largest species of turtle?",
"Who invented the ATM?",
"What is the boiling point of helium in Celsius?",
"How many players are on a volleyball team?",
"What is the largest cave system in the world?",
"Who wrote The Sound and the Fury?",
"What is the chemical symbol for lead?",
"How many ribs does a human typically have?",
"What is the fastest marine animal?",
"Who discovered the double helix structure of DNA?",
"What is the main ingredient in tzatziki?",
"How many players are on a handball team?",
"What is the largest peninsula in the world?",
"Who composed The Nutcracker?",
"What is the smallest country in Africa?",
"How many squares are on a chessboard?",
"What is the largest species of crocodile?",
"Who invented the microwave oven?",
"What is the main ingredient in paella?",
"How many valves does the human heart have?",
"What is the deepest cave in the world?",
"Who wrote The Grapes of Wrath?",
"What is the chemical symbol for platinum?",
"How many players are on an ice hockey team?",
"What is the largest species of bat?",
"Who discovered the proton?",
"What is the primary ingredient in gua bao?",
"How many degrees are in an equilateral triangle?",
"What is the longest canal in the world?",
"Who composed Clair de Lune?",
"What is the smallest continent by land area?",
"How many segments are in an insect's body?",
"What is the largest species of octopus?",
"Who invented the helicopter?",
"What is the main ingredient in ratatouille?",
"How many players are on a lacrosse team?",
"What is the highest cliff in the world?",
"Who wrote Les Misérables?",
"What is the chemical symbol for magnesium?",
"How many chambers does a crocodile's heart have?",
"What is the largest inland sea in the world?",
"Who discovered electromagnetism?",
"What is the primary ingredient in ceviche?",
"How many players are on a field hockey team?",
"What is the longest tunnel in the world?",
"Who composed The Four Seasons?",
"What is the smallest bone in the ear?",
"How many spikes are on the Statue of Liberty's crown?",
"What is the largest species of antelope?",
"Who invented the thermometer?",
"What is the main ingredient in risotto?",
"How many players are on a polo team?",
"What is the highest dam in the world?",
"Who wrote A Tale of Two Cities?",
"What is the chemical symbol for chromium?",
"How many pairs of wings do bees have?",
"What is the largest species of jellyfish?",
"Who discovered the structure of the atom?",
"What is the primary ingredient in sashimi?",
"How many players are on a curling team?",
"What is the longest mountain range in the world?",
"Who composed The Barber of Seville?",
"What is the smallest planet by mass?",
"How many bones are in the human hand?",
"What is the largest species of owl?",
"Who invented the escalator?",
"What is the main ingredient in gazpacho?",
"How many players are on a dodgeball team?",
"What is the highest plateau in the world?",
"Who wrote The Sun Also Rises?",
"What is the chemical symbol for manganese?",
"How many tentacles does a jellyfish typically have?",
"What is the largest species of rhinoceros?",
"Who discovered the photoelectric effect?",
"What is the primary ingredient in baklava?",
"How many players are on a synchronized swimming team?",
"What is the longest strait in the world?",
"Who composed Rhapsody in Blue?",
"What is the smallest unit of energy?",
"How many vertebrae are in the human spine?",
"Who is Batman?",
"Recipe for chocolate cake?",
"Distance to the Moon?",
"How does a black hole form?",
"What is the speed of light?",
"How do airplanes fly?",
"What causes earthquakes?",
"How does the immune system work?",
"What is quantum entanglement?",
"How is glass made?",
"Why is the sky blue?",
"How do vaccines work?",
"What is the theory of relativity?",
"How do plants make food?",
"What is DNA?",
"How does Wi-Fi work?",
"What is the tallest mountain on Earth?",
"How do tides work?",
"What is the Fibonacci sequence?",
"How does a combustion engine work?",
"What is dark matter?",
"How does electricity work?",
"What is machine learning?",
"How do hurricanes form?",
"What is the Big Bang theory?",
"How does sound travel?",
"What is a meme?",
"How do satellites stay in orbit?",
"What is the greenhouse effect?",
"How does the human brain store memories?",
"What is blockchain?",
"How does sonar work?",
"What is CRISPR?",
"How do noise-canceling headphones work?",
"What is entropy?",
"How does a nuclear reactor work?",
"What is the placebo effect?",
"How do solar panels work?",
"What is a supernova?",
"How does the stock market work?",
"What is photosynthesis?",
"How do lithium batteries work?",
"What is the Turing test?",
"How does carbon dating work?",
"What is a neural network?",
"How does the internet work?",
"What is tectonic plate movement?",
"How do antibiotics work?",
"What is a wormhole?",
"How does inflation work?",
"What is the ozone layer?",
"How does a transformer model work?",
"What is general relativity?",
"How does blood clotting work?",
"What is a quasar?",
"How do electric motors work?",
"What is the human microbiome?",
"How does a gyroscope work?",
"What is the Drake equation?",
"How does anesthesia work?",
"What is a Fourier transform?",
"How do wind turbines generate power?",
"What is epigenetics?",
"How does a submarine dive?",
"What is chaos theory?",
"How do fiber optics work?",
"What is stem cell therapy?",
"How does a jet engine work?",
"What is the multiverse theory?",
"How do 3D printers work?",
"What is neuroplasticity?",
"How does GPS work?",
"What is the doppler effect?",
"How do plants communicate?",
"What is antimatter?",
"How does a MRI machine work?",
"What is the prisoner's dilemma?",
"How does fermentation work?",
"What is a Dyson sphere?",
"How do octopuses change color?",
"What is the uncanny valley?",
"How does osmosis work?",
"What is a Faraday cage?",
"How do birds navigate during migration?",
"What is the tragedy of the commons?",
"How does a telescope work?",
"What is game theory?",
"How do muscles grow?",
"What is a neutron star?",
"How does sourdough bread rise?",
"What is the butterfly effect?",
"How do whales communicate?",
"What is a Lagrange point?",
"How does radioactive decay work?",
"What is the observer effect in physics?",
"How do coral reefs form?",
"What is a Ponzi scheme?",
"How does smell work?",
"What is the Heisenberg uncertainty principle?",
"How do languages evolve?",
"What is a solar flare?",
"How does hibernation work?",
"What is the trolley problem?",
"How do crystals form?",
"What is a logic gate?",
"How does the ear perceive sound?",
"What is the Mandela effect?",
"How do tornadoes form?",
"What is a mRNA vaccine?",
"How does erosion shape landscapes?",
"What is dunning-kruger effect?",
"How do computers store data?",
"What is a geomagnetic storm?",
"How do fungi reproduce?",
"What is a cognitive bias?",
"How does echolocation work?",
"What is the Kardashev scale?",
"How do tears form?",
"What is spectroscopy?",
"How does a lie detector work?",
"What is the difference between fission and fusion?",
"How does the kidney filter blood?",
"What is quantum computing?",
"How do spiders produce silk?",
"What is the social contract theory?",
"How does a compass work?",
"What is the Coriolis effect?",
"How do cells divide?",
"What is a paradigm shift?",
"How does aging work biologically?",
"What is a dead star?",
"How do avalanches start?",
"What is the Rorschach test?",
"How does taste work?",
"What is the Monty Hall problem?",
"How do vaccines create herd immunity?",
"What is a semiconductor?",
"How does the liver detoxify?",
"What is superposition in quantum mechanics?",
"How do wildfires spread?",
"What is the anthropic principle?",
"How does caffeine affect the brain?",
"What is a geothermal vent?",
"How does a lock and key mechanism work in enzymes?",
"What is the Sapir-Whorf hypothesis?",
"How do cold fronts form?",
"What is a qubit?",
"How does the lymphatic system work?",
"What is Occam's razor?",
"How do languages die?",
"What is a black dwarf?",
"How does color vision work?",
"What is the hot hand fallacy?",
"How does wind form?",
"What is Maslow's hierarchy of needs?",
"How do gut bacteria affect mood?",
"What is a Möbius strip?",
"How does lightning form?",
"What is the sunk cost fallacy?",
"How do animals go extinct?",
"What is the second law of thermodynamics?",
"How does a compiler work?",
"What is the speed of sound?",
"How do hormones regulate the body?",
"What is confirmation bias?",
"How does a seismograph work?",
"What is the law of large numbers?",
"How do immune cells identify pathogens?",
"What is spontaneous combustion?",
"How does color mixing work in light vs paint?",
"What is the cosmological constant?",
"How does a centrifuge work?",
"What is survivorship bias?",
"How do planets form?",
"What is operant conditioning?",
"How does a chemical bond form?",
"What is the semantic web?",
"How does fog form?",
"What is bioaccumulation?",
"How does a pulsar work?",
"What is regression to the mean?",
"How does rust form?",
"What is zero-point energy?",
"How do tsunamis form?",
"What is the Peter principle?",
"How does a capacitor work?",
"What is Moore's law?",
"How do trees grow so tall?",
"What is the observer selection effect?",
"How does a heat pump work?",
"What is Bayes' theorem?",
"How do parasites control host behavior?",
"What is the gambler's fallacy?",
"How does deforestation affect rainfall?",
"What is the Copenhagen interpretation?",
"How does a geiger counter work?",
"What is symbolic logic?",
"How do planets stay in orbit?",
"What is the Pareto principle?",
"How does sonar detect objects?",
"What is Schrödinger's cat?",
"How does sleep affect memory consolidation?",
"What is tidal locking?",
"How does carbon capture work?",
"What is the Flynn effect?",
"How do volcanoes erupt?",
"What is an algorithm?",
"What is the most memorable dream you've ever had?",
"If you could live in any fictional world, which one would it be?",
"What is a skill you've always wanted to learn?",
"If you could have dinner with any historical figure, who would it be?",
"What is the best advice you've ever received?",
"If you could travel to any time period, when would you go?",
"What is a book that changed your life?",
"If you could have any superpower, what would it be?",
"What is a movie you could watch over and over again?",
"If you could live anywhere in the world, where would it be?",
"What is a hobby you've always wanted to try?",
"If you could change one thing about the world, what would it be?",
"What is the best gift you've ever received?",
"If you could meet any fictional character, who would it be?",
"What is a song that always makes you feel happy?",
"If you could create a new holiday, what would it be?",
"What is the best meal you've ever had?",
"If you could be any animal for a day, what would it be?",
"What is a place you've always wanted to visit?",
"If you could have any job in the world, what would it be?",
"What is a goal you've always wanted to achieve?",
"If you could speak any language fluently, what would it be?",
"What is a memory you will always cherish?",
"If you could redesign the human body, what would you change?",
"What is a topic you could talk about for hours?",
"If you could live to be 100 years old but be broke or die at 50 but be rich, which would you choose?",
"What is a word that you find beautiful?",
"If you could have any item of clothing, what would it be?",
"What is a sound that you find calming?",
"If you could be a character in a movie, who would you be?",
"What is a scent that reminds you of your childhood?",
"If you could eat only one thing for the rest of your life, what would it be?",
"What is a question you've always wanted to ask?",
"If you could control one element, what would it be?",
"What is a habit you're trying to break?",
"If you could be an expert in any field, what would it be?",
"What is a quote that inspires you?",
"If you could change your name, what would it be?",
"What is a lesson you've learned the hard way?",
"If you could have any pet in the world, what would it be?",
"What is a tradition you'd like to start?",
"If you could know the answer to any question, what would it be?",
"What is a risk you've taken that paid off?",
"If you could make one thing in the world disappear, what would it be?",
"What is a fear you've overcome?",
"If you could change the past, what would you change?",
"What is a joy in your life?",
"If you could create a new type of art, what would it be?",
"What is a dream you're working towards?",
"If you could ask one question to a psychic, what would it be?"
]

In [ ]:
prefix = (
    "You are an expert Physiotherapist AI assistant for the RehabQuest platform. "
    "Your role is to provide concise, evidence-based rehabilitation guidance "
    "grounded in peer-reviewed clinical literature and WHO recommendations. "
    "Always cite relevant studies when possible. Limit answers to three paragraphs. "
    "Use metric units. If the question is outside your expertise, state so clearly. "
    "RehabQuest is a healthcare startup specialising in AI-driven musculoskeletal "
    "rehabilitation using computer vision and wearable sensors. The platform tracks "
    "patient exercises in real time, provides corrective feedback, and generates "
    "progress reports for clinicians. You must adhere to HIPAA guidelines and never "
    "provide a definitive diagnosis. Always recommend that patients consult their "
    "treating physician for personalised medical advice before changing their "
    "rehabilitation programme. Respond in professional but approachable language. "
    "Now answer the following clinical question. "
)

prompts = [
    prefix + "What exercises help with lower back pain?",
    prefix + "Is applying ice effective for reducing swelling?",
    prefix + "Define correct sitting posture for office workers.",
    prefix + "How long should a rotator cuff tear rehabilitation programme last?",
    prefix + "What is the recommended rest period after an acute ankle sprain?",
    prefix + "Which stretches are most effective for tight hamstrings?",
    prefix + "How should a patient progress from non-weight-bearing to full weight-bearing after a tibial fracture?",
    prefix + "What is the role of proprioception training after ACL reconstruction?",
    prefix + "How many sets and reps are recommended for strengthening the quadriceps post knee surgery?",
    prefix + "What are the signs that a patient is overtraining during rehabilitation?",
    prefix + "How effective is dry needling for myofascial pain syndrome?",
    prefix + "What is the difference between active and passive physiotherapy?",
    prefix + "When is it safe to return to sport after a hamstring strain?",
    prefix + "How does ultrasound therapy aid soft tissue healing?",
    prefix + "What are the best exercises for strengthening the hip abductors?",
    prefix + "How should breathing be coordinated during core stability exercises?",
    prefix + "What is the McKenzie method and when is it indicated?",
    prefix + "How do wearable sensors improve rehabilitation outcomes?",
    prefix + "What is the recommended frequency of physiotherapy sessions for chronic neck pain?",
    prefix + "How does foam rolling affect muscle recovery?",
    prefix + "What are the early mobilisation protocols after total knee replacement?",
    prefix + "How should a patient warm up before starting rehabilitation exercises?",
    prefix + "What is the evidence for kinesiology taping in shoulder impingement?",
    prefix + "How does sleep quality affect musculoskeletal recovery?",
    prefix + "What exercises are contraindicated after lumbar discectomy?",
    prefix + "How is gait analysis used in rehabilitation planning?",
    prefix + "What is the role of hydrotherapy in post-surgical rehabilitation?",
    prefix + "How long does it typically take to recover from a grade 2 ligament sprain?",
    prefix + "What are the benefits of eccentric training for tendinopathy?",
    prefix + "How should rehabilitation differ for elderly patients with hip fractures?",
    prefix + "What is the Oswestry Disability Index used for?",
    prefix + "How can computer vision detect incorrect squat form?",
    prefix + "What are the clinical criteria for diagnosing patellofemoral pain syndrome?",
    prefix + "How does chronic pain affect rehabilitation adherence?",
    prefix + "What is the recommended load progression for Achilles tendinopathy rehab?",
    prefix + "How effective is TENS therapy for post-operative pain management?",
    prefix + "What is the difference between isometric and isotonic exercises?",
    prefix + "How should rehabilitation be modified for a diabetic patient with peripheral neuropathy?",
    prefix + "What are the red flags in low back pain that require immediate referral?",
    prefix + "How does obesity affect joint loading during rehabilitation exercises?",
    prefix + "What is neuromuscular electrical stimulation and when is it used?",
    prefix + "How can a patient self-monitor exercise intensity during home rehabilitation?",
    prefix + "What are the stages of tissue healing and how do they guide treatment?",
    prefix + "How effective is spinal manipulation for non-specific low back pain?",
    prefix + "What is the role of the transverse abdominis in lumbar stability?",
    prefix + "How does stress and anxiety impact musculoskeletal pain perception?",
    prefix + "What is the minimal detectable change for the Visual Analogue Scale in pain assessment?",
    prefix + "How should rehabilitation be adapted for a patient with osteoporosis?",
    prefix + "What are the best outcome measures for tracking shoulder rehabilitation progress?",
    prefix + "How does dehydration affect muscle performance during exercise?",
    prefix + "What is the evidence for Pilates in chronic low back pain rehabilitation?",
    prefix + "How long should static stretches be held for optimal flexibility gains?",
    prefix + "What are the principles of graded exposure in pain rehabilitation?",
    prefix + "How does posture affect cervicogenic headaches?",
    prefix + "What is the recommended rehabilitation protocol after a Bankart repair?",
    prefix + "How can wearable accelerometers quantify patient activity outside clinic sessions?",
    prefix + "What is the difference between a muscle strain and a muscle tear?",
    prefix + "How should a patient manage delayed onset muscle soreness?",
    prefix + "What exercises help prevent falls in older adults?",
    prefix + "How does manual therapy compare to exercise therapy for neck pain?",
    prefix + "What is the role of scapular stabilisation exercises in shoulder rehabilitation?",
    prefix + "How is range of motion measured and documented in clinical practice?",
    prefix + "What are the functional milestones for return to running after a calf tear?",
    prefix + "How does vitamin D deficiency affect musculoskeletal health?",
    prefix + "What is the clinical significance of muscle imbalance in the lower limb?",
    prefix + "How effective is shockwave therapy for plantar fasciitis?",
    prefix + "What are the biomechanical risk factors for developing runner's knee?",
    prefix + "How should a rehabilitation programme be structured for a sedentary office worker with chronic shoulder pain?",
    prefix + "What is the role of balance boards in ankle rehabilitation?",
    prefix + "How does fear-avoidance behaviour prolong musculoskeletal recovery?",
    prefix + "What is the recommended progression for plantar fasciitis rehabilitation?",
    prefix + "How do real-time corrective cues improve exercise performance?",
    prefix + "What are the clinical features of frozen shoulder and how is it managed?",
    prefix + "How should rehabilitation differ between contact and non-contact sports injuries?",
    prefix + "What is the evidence for lumbar support belts in preventing back injury?",
    prefix + "How does anti-inflammatory medication interact with tendon healing?",
    prefix + "What are the best exercises for strengthening the gluteus medius?",
    prefix + "How is functional movement screening used in injury prevention?",
    prefix + "What is the role of ergonomics in preventing repetitive strain injuries?",
    prefix + "How effective is cold water immersion for athlete recovery?",
    prefix + "What are the stages of ACL rehabilitation and their timelines?",
    prefix + "How does a herniated disc differ from spinal stenosis in terms of rehabilitation?",
    prefix + "What is the evidence for Nordic hamstring curls in injury prevention?",
    prefix + "How should rehabilitation goals be set collaboratively with patients?",
    prefix + "What is the role of compression garments in managing oedema?",
    prefix + "How does cardiovascular fitness affect rehabilitation outcomes?",
    prefix + "What exercises are recommended for thoracic spine mobility?",
    prefix + "How is clinical reasoning applied when designing a rehabilitation programme?",
    prefix + "What is the difference between open and closed kinetic chain exercises?",
    prefix + "How do wearable EMG sensors help assess muscle activation patterns?",
    prefix + "What is the recommended approach for managing chronic Achilles tendinopathy?",
    prefix + "How should a patient with hypermobility syndrome approach strength training?",
    prefix + "What are the physiological adaptations expected after 8 weeks of resistance training?",
    prefix + "How is pain catastrophising measured and addressed in rehabilitation?",
    prefix + "What is the evidence for yoga as a complement to physiotherapy?",
    prefix + "How does load management prevent stress fractures in runners?",
    prefix + "What are the key principles of the biopsychosocial model in physiotherapy?",
    prefix + "How effective is heat therapy compared to cold therapy for muscle spasm?",
    prefix + "What are the indications and contraindications for traction therapy?",
    prefix + "How should a rehabilitation programme be designed for a post-mastectomy patient with lymphoedema?",
    prefix + "What is the minimal clinically important difference for the DASH outcome measure?",
]

In [ ]:
[
"What are the core vision and mission statements of Tongaat Hulett? ",
"Summarize the geographic footprint of Tongaat Hulett's operations across South Africa, Mozambique, Zimbabwe, and Botswana. ",
"What was the total volume of sugar produced by Tongaat Hulett in the 2021 financial year? ",
"How many people were employed by Tongaat Hulett at the peak of the sugar milling season in 2021? ",
"Detail the percentage of Board members at Tongaat Hulett who were non-executive and independent in 2021. ",
"Explain the Manufactured and Financial capitals as described in Tongaat Hulett's business model. ",
"What were the key focus areas for Tongaat Hulett in 2022 to enable high performance and operational excellence? ",
"Summarize the impact of the COVID-19 pandemic on Tongaat Hulett's operations and workforce during 2021. ",
"How much did Tongaat Hulett invest in COVID-19 avoidance, mitigation, and treatment in 2021? ",
"Describe the role and responsibilities of the Social, Ethics, Health and Safety Committee at Tongaat Hulett. ",
"What is Tongaat Hulett's target for energy intensity reduction by the year 2025? ",
"List the market-leading brands for sugar and animal feeds mentioned in the Tongaat Hulett report. ",
"What was the total socio-economic development (SED) expenditure by Tongaat Hulett in 2021? ",
"Detail the change in Tongaat Hulett's scope 1 and scope 2 carbon emissions between 2020 and 2021. ",
"How does Tongaat Hulett define and manage its Intellectual Capital? ",
"What were the primary environmental efficiency investments made by Tongaat Hulett in 2021? ",
"Summarize the findings of the 2021 corporate reputation survey conducted by Tongaat Hulett. ",
"What are the specific targets for water efficiency improvement at Tongaat Hulett by 2025? ",
"Explain the relationship between Tongaat Hulett and its small-scale growers, including the volume of sugarcane supplied. ",
"What was the Lost Time Injury Frequency Rate (LTIFR) for Tongaat Hulett in 2021? ",
"Describe Tongaat Hulett's approach to human rights and child labor in its supply chain. ",
"What were the total volumes of hazardous and non-hazardous waste disposed of by Tongaat Hulett in 2021? ",
"List the third-party certifications held by Tongaat Hulett operations, such as ISO 45001. ",
"How does Tongaat Hulett align its business activities with the UN Sustainable Development Goals? ",
"What was the total revenue generated by Tongaat Hulett for the 2021 financial year? ",
"Explain the significance of the Sugar Industry Masterplan for Tongaat Hulett's South African operations. ",
"Detail the training and development spend for employees at Tongaat Hulett in 2021. ",
"What are the primary climate change risks identified and prioritized by Tongaat Hulett? ",
"Summarize the Success Management programme used for performance management at Tongaat Hulett. ",
"Who provides the independent external assurance for Tongaat Hulett's ESG and Climate Change Report? ",
"What is the stated purpose of Implats according to the 2023 ESG report? ",
"Name the managed operations included in the scope of the Implats 2023 ESG report. ",
"What was the Lost Time Injury Frequency Rate (LTIFR) for the Implats Group in 2023? ",
"Detail the total value distributed by Implats to its stakeholders in the 2023 financial year. ",
"What percentage of the Implats Board of Directors identifies as female? ",
"Summarize the Group CEO's statement on Implats' safety performance and the goal of zero harm. ",
"What are the three pillars of the Implats ESG framework? ",
"How much did Implats invest in socio-economic development and community projects in 2023? ",
"Explain the double materiality principle used in the preparation of Implats' ESG reports. ",
"What are the primary metals produced by Implats and their specific uses in modern technology? ",
"Describe the progress and intended impact of the 35MW solar PV project at Zimplats. ",
"What was the total water recycling and reuse rate achieved by the Implats Group in 2023? ",
"Detail the We Care programme's support for the families of employees who lost their lives in fatal incidents. ",
"What were the key outcomes for Employees in terms of wages and benefits paid by Implats in 2023? ",
"Explain the PS3 strategy and its focus on sustainability alignment at Implats. ",
"What is Implats' specific target for reducing its carbon emissions by 2030? ",
"Describe the progress of the sulphur dioxide (SO2) abatement technology installation at Zimplats. ",
"What were the external ESG ratings received by Implats from agencies like MSCI and S&P Global in 2023? ",
"How does Implats manage air quality and reduce particulate matter emissions across its operations? ",
"Summarize the importance of the Royal Bafokeng Platinum (RBPlat) acquisition for Implats' Western Limb assets. ",
"What is the IRMA audit and when does Implats plan to conduct it at a managed site? ",
"Detail the roles and responsibilities of the Social, Transformation and Remuneration (STR) Committee at Implats. ",
"How does Implats address the issue of gender-based violence (GBV) in its host communities? ",
"What was the total direct and indirect energy consumption for the Implats Group in 2023? ",
"Describe the tailings remining project at Impala Rustenburg and its contribution to the circular economy. ",
"What are the specific occupational health milestones for the South African mining industry that Implats is working toward? ",
"How does Implats ensure business ethics and maintain a zero-tolerance stance on fraud and corruption? ",
"Detail the environmental legal compliance status of Implats' managed operations in 2023. ",
"What are the Tier 1 host communities identified for Impala Rustenburg and Marula mine? ",
"Summarize the Sustaining Livelihoods framework and its four key focus areas at Implats. ",
"Compare the water efficiency targets and recycling performance reported by Tongaat Hulett and Implats. ",
"Contrast the carbon emission reduction strategies of Tongaat Hulett and Implats based on their respective industry contexts. ",
"How do both companies approach stakeholder engagement and what are their primary communication channels? ",
"Analyze the differences in how Tongaat Hulett and Implats report on human capital and workforce diversity. ",
"Compare the governance structures of the two companies, specifically the focus of their board-level ESG committees. ",
"How do both organizations integrate the United Nations Global Compact principles into their respective reporting? ",
"Contrast the socio-economic development (SED) spending and focus areas of Tongaat Hulett and Implats. ",
"Compare the safety performance metrics, such as LTIFR and fatalities, of both companies over the reported periods. ",
"How do Tongaat Hulett and Implats address the physical impacts of climate change, such as droughts or floods, on their operations? ",
"Analyze the role of renewable energy (solar, hydro, etc.) in the decarbonization plans of both companies. ",
"What are the inputs and outcomes of Intellectual Capital as described in both the Tongaat Hulett and Implats ESG reports? ",
"How do both companies manage waste, and what are their respective targets for landfill diversion? ",
"Compare the approach to Double Materiality in the Implats report with the materiality determination process in the Tongaat Hulett report. ",
"How have both companies responded to the COVID-19 pandemic in terms of community support and employee health? ",
"Discuss the use of external assurance providers (BDO, Nexia SAB&T, etc.) in the ESG reporting of both companies. ",
"What were the specific Human Capital challenges, such as restructuring, encountered by Tongaat Hulett in 2021? ",
"Compare the Business Model illustrations and the 'capitals' approach used by both Tongaat Hulett and Implats. ",
"How do both reports address Human Rights and child labor within their respective supply chains? ",
"Contrast the production capacity and revenue of Tongaat Hulett with the production and value distribution of Implats. ",
"Discuss the significance of site-specific versus group-level ESG policies in both companies. ",
"What were the specific reasons for restating prior year data in the Tongaat Hulett 2021 ESG report? ",
"Explain the TCFD Application Table provided in the Tongaat Hulett report and its importance. ",
"Detail the causes of the 20 employee deaths at Tongaat Hulett in 2021, distinguishing between work-related and other causes. ",
"Describe the Success Management values and the self-management practices implemented at Tongaat Hulett. ",
"What are the environmental legal and regulatory compliance procedures for Tongaat Hulett's operational sites? ",
"Summarize the Sustainability Management Framework and the data management systems used by Tongaat Hulett. ",
"Detail the breakdown of sugarcane volumes supplied to Tongaat Hulett mills from different types of farmers. ",
"What was the Total Recordable Injury Frequency Rate (TRIFR) for Tongaat Hulett between 2018 and 2021? ",
"Explain the Material adverse change (MAC) dispute regarding the sale of Tongaat Hulett's starch business. ",
"What are the specific outcomes for Transformation as reported in Tongaat Hulett's 2021 business model? ",
"Describe the eight-stage stakeholder engagement approach used by Implats to ensure participatory management. ",
"What are the specific components and focus areas of the PS3 Strategy at Implats? ",
"Detail the 2023 safety milestones, such as millionaire status shifts, achieved by Implats operations. ",
"Explain the involuntary turnover statistics for the Implats Group in 2023 across its different operating regions. ",
"What were the findings of the independent tailings review board regarding Implats' storage facilities in 2023? ",
"Describe the Gender-based Violence (GBV) response fund and the community partnerships initiated by Implats. ",
"What is the Global Industry Standard on Tailings Management (GISTM) and what is Implats' roadmap for compliance? ",
"Detail the Enterprise and Supplier Development (ESD) initiatives and spend at Implats' South African operations. ",
"How does Implats use woodchips in the concurrent rehabilitation of its tailings dam side slopes? ",
"What was the total amount of Coal, Diesel, and Electricity consumed by the Implats Group in 2023? ",
"Explain the inward-focused financial materiality versus outward-focused impact materiality at Implats. ",
"Describe the We Care programme's specific support for families of deceased employees, including education aid. ",
"What were the specific Lowlights and Challenges reported by Implats in its 2023 governance and social sections? ",
"Summarize the progress of the sulphur dioxide abatement project at Zimplats as of the end of the 2023 financial year. ",
"What are the Six Capitals mapped in the Implats report and how are they used to illustrate value creation? ",
"Analyze the Human Capital purpose statement of Tongaat Hulett and its link to organizational potential. ",
"How does Tongaat Hulett monitor and report on environmental incidents and complaints across its operations? ",
"Detail the investments made by Tongaat Hulett in projects to improve environmental efficiencies in 2021. ",
"Explain the significance of Ethical Leadership and Purpose-driven Governance in the Implats ESG report. ",
"Describe the Huletts brand refresh and the 'Humthem' campaign mentioned in the Tongaat Hulett report. ",
"What are the Material Matters grouped into 10 themes for value creation identified by Implats? ",
"Summarize the general Approach to Sustainability at Tongaat Hulett as described in the 2021 report. ",
"What was the total number of person days lost due to industrial action at Tongaat Hulett between 2018 and 2021? ",
"Detail the Mineral Waste Management strategy at Implats and the focus on mineral residue recovery. ",
"How does Tongaat Hulett define and report on Near Misses and High Frequency Risk Incidents (HFRIs)? ",
"What is the NOSA 5 Star system and why was its certification discontinued at Tongaat Hulett? ",
"Describe the Southern Lodestar Foundation partnership and the nutrition programs supported by Tongaat Hulett. ",
"What was the CSI/SED spend as a percentage of net profit after tax for Tongaat Hulett in 2021? ",
"Explain the PS3 Focus Areas for Stakeholder Relationship Management at Implats. ",
"What were the specific climate-related hazards Tongaat Hulett aimed to combat through its 2021 strategy? ",
"Detail the education and skills development projects supported by Implats in South Africa during 2023. ",
"Summarize the Message from the Chairperson of the SEHSC regarding safety culture at Tongaat Hulett. ",
"What are the Managed Entities versus Non-managed Entities in the Implats Group reporting scope? ",
"How does Tongaat Hulett address Food Safety and what are its current plans for ISO certification? ",
"Explain the Decarbonization and energy security policy statement endorsed by the Implats board. ",
"What was the total volume of waste recycled by Tongaat Hulett in 2021, and what was the goal? ",
"Describe the small-scale farmer support initiatives and the Jobs Fund partnership at Tongaat Hulett. ",
"What are the external ESG rankings and Yearbook inclusions for Implats in 2023? ",
"Summarize the Land capital section for Tongaat Hulett, including EIA approved development land. ",
"Detail the Occupational Health incidents, reversible and irreversible, for Tongaat Hulett in 2021. ",
"How does Implats report on its human rights impact assessment and alignment with the UNVPSHR? ",
"What was the net electricity purchased versus self-generated electricity for Tongaat Hulett in 2021? ",
"Explain the mine closure and rehabilitation financial provisions for Implats in 2023. ",
"Describe the Success Management values, such as 'Safely home every day', at Tongaat Hulett. ",
"What are the Key Initiatives 2021 for Social and Relationship Capital at Tongaat Hulett? ",
"Summarize the CEO statement at Implats regarding the 'just transition' to clean energy. ",
"What was the total remuneration paid by Tongaat Hulett in the 2021 financial year? ",
"Detail the water abstracted and used for irrigation by Tongaat Hulett's agricultural operations. ",
"How does Implats manage Conflict of Interest and Corruption cases as reported in its 2023 ethics section? ",
"Describe the Huletts SunSweet and Equisweet brands mentioned in the Tongaat Hulett report. ",
"What are the Coverage Areas for Tongaat Hulett's medium-term ESG strategy? ",
"Explain the materiality assessment process, including relevance and importance, at Implats. ",
"Summarize the Health and Safety commitments of Tongaat Hulett, focusing on the zero harm goal. ",
"What was the total volume of water output for Tongaat Hulett's mills and factories in 2021? ",
"Detail the total indirect energy consumption for the Tongaat Hulett group in 2021. ",
"How does Implats report on employee engagement through the 'Care and Growth' programme? ",
"Describe the Sustainability Management framework approval process at Tongaat Hulett. ",
"What was the raw sugar versus refined sugar production for Tongaat Hulett in the 2021 period? ",
"Summarize the Intellectual Property (IP) protection methods used by Tongaat Hulett. ",
"What are the future priorities for Natural Capital at Tongaat Hulett as stated in the 2021 report? "
]

In [ ]:
base_history = "User: Hello AI.\nAssistant: Hi there! How can I help you today?\n"

questions = [
    "Can you explain how the RehabQuest pose tracking works in detail?",
    "What is the calibration procedure used at the start of each session?",
    "How does the T-pose calibration normalise body proportions across different patients?",
    "What are the hardware requirements for real-time pose tracking?",
    "How does the MediaPipe holistic model detect the 33 pose landmarks?",
    "What is the role of cosine similarity in computing joint angles?",
    "How are joint angles computed in three dimensions using vector mathematics?",
    "How is the system validated against the Vicon motion capture gold standard?",
    "What accuracy metrics are used to evaluate pose tracking performance?",
    "How does camera distance affect the accuracy of landmark detection?",
    "What is the minimum GPU specification needed for real-time processing?",
    "How does the system handle occlusion when body parts are partially hidden?",
    "What frame rate is required to achieve clinically acceptable motion tracking?",
    "How are left and right side landmarks differentiated in the holistic model?",
    "What happens if the T-pose calibration is performed incorrectly?",
    "How does the system account for varying patient heights and limb lengths?",
    "Can the pose tracking work with a standard RGB webcam or does it need depth sensors?",
    "How are the 33 landmarks mapped to anatomical joint definitions?",
    "What filtering or smoothing is applied to raw landmark coordinates?",
    "How does the system detect and reject outlier frames during a session?",
    "How is shoulder flexion angle specifically calculated from landmark vectors?",
    "How is knee extension range of motion extracted from the landmark data?",
    "What is the typical latency from movement to on-screen feedback?",
    "How does lighting condition affect landmark detection confidence scores?",
    "What confidence threshold is used to accept or reject a detected landmark?",
    "How does the system track spinal alignment and posture during exercises?",
    "How are exercise repetitions counted from the joint angle time series?",
    "What machine learning model underlies the MediaPipe pose estimator?",
    "How was the MediaPipe model trained and what datasets were used?",
    "Can the system distinguish between correct and compensatory movement patterns?",
    "How is data from multiple sessions stored and compared over time?",
    "What data format is used to export session results for clinician review?",
    "How does the system perform on patients with limb prosthetics or assistive devices?",
    "What are the known failure modes of the cosine similarity angle computation?",
    "How is the world coordinate frame defined relative to the camera?",
    "How does the system handle patients who cannot perform the initial T-pose?",
    "What is the mean absolute error reported against the Vicon gold standard?",
    "How does body-worn clothing affect the accuracy of landmark detection?",
    "Can multiple cameras be used simultaneously to improve tracking accuracy?",
    "How are upper and lower extremity exercises treated differently in the pipeline?",
    "What happens to tracking accuracy when the patient moves out of the camera frame?",
    "How is the skeleton model re-initialised after a tracking loss event?",
    "How are hip joint angles computed and which landmarks are used?",
    "What is the difference between 2D and 3D landmark coordinates in MediaPipe?",
    "How does the system calculate symmetry scores between left and right sides?",
    "What network architecture is used for the pose landmark regression?",
    "How are progress reports generated from accumulated session data?",
    "Can the system operate offline without an internet connection?",
    "How is patient privacy protected when storing video and landmark data?",
    "What are the planned future improvements to the pose tracking pipeline?",
]

prompts_50 = []
history = base_history

for i, question in enumerate(questions, 1):
    history += f"User: {question}\n"
    history += f"Assistant: Here is my detailed answer for turn {i}.\n"
    prompts_50.append(history + f"User: Can you elaborate further on your answer about: {question}")

labels_50 = [f"Turn {i}" for i in range(1, 51)]